# Tarea 22 · Codificación de variables categóricas

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/julianschmidt1/tpi-analisis-de-datos/blob/feat/22-codificacion/notebooks/trabajo/22_codificacion.ipynb)

**TPI · Introducción al Análisis de Datos · UTN FRA · División 343**  
**Integrantes:** Matías, Julián y Emmanuel. **Responsable:** Emmanuel.

Entrada: los cuatro CSV de `data/processed/particion/` generados en la tarea 21. Aplicamos One-Hot a las variables nominales y codificación ordinal a `Contract`. Exportamos los predictores codificados, todavía sin escalar, en `data/processed/codificacion/`. Las etiquetas no cambian.

## Cómo ejecutarlo

- **Local:** instalar `requirements.txt` y ejecutar todas las celdas desde cualquier carpeta dentro del repositorio.
- **Colab:** abrir el badge cuando la rama esté publicada o subir el notebook. Cargar los cuatro CSV de la tarea 21 en `data/processed/particion/` dentro del directorio de trabajo de Colab. Si ya están publicados, también se pueden obtener desde GitHub, conservando esa estructura. No generamos otra partición.
- Las salidas aparecen en `data/processed/codificacion/`; en Colab están en el panel **Archivos**. Al integrar el notebook, cambiar el badge a `main`.


## 1. Carga de la partición existente

Conservamos `fila_origen` como índice para alinear clientes y etiquetas; no lo usamos como predictor. Verificamos que los conjuntos sean disjuntos y que no contengan el objetivo entre los predictores.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

try:
    import google.colab
except ImportError:
    EN_COLAB = False
else:
    EN_COLAB = True

actual = Path.cwd().resolve()
RAIZ = actual if EN_COLAB else next(
    (ruta for ruta in (actual, *actual.parents)
     if (ruta / 'PLAN_TAREAS.md').is_file()), None)
if RAIZ is None:
    raise FileNotFoundError('Abrí Jupyter dentro del repositorio del TPI.')
entrada = RAIZ / 'data' / 'processed' / 'particion'
archivos = ['X_train', 'X_test', 'y_train', 'y_test']
faltantes = [nombre for nombre in archivos if not (entrada / f'{nombre}.csv').is_file()]
if faltantes:
    raise FileNotFoundError(f'Faltan {faltantes} en {entrada}. Ejecutá la tarea 21 o copiá sus salidas.')
X_train = pd.read_csv(entrada / 'X_train.csv', index_col='fila_origen')
X_test = pd.read_csv(entrada / 'X_test.csv', index_col='fila_origen')
y_train = pd.read_csv(entrada / 'y_train.csv', index_col='fila_origen')['Churn']
y_test = pd.read_csv(entrada / 'y_test.csv', index_col='fila_origen')['Churn']

assert X_train.index.is_unique and X_test.index.is_unique
assert X_train.index.intersection(X_test.index).empty
assert X_train.index.equals(y_train.index) and X_test.index.equals(y_test.index)
assert X_train.columns.equals(X_test.columns)
assert 'Churn' not in X_train.columns
assert not X_train.isna().any().any() and not X_test.isna().any().any()
print(f'Train: {X_train.shape}; test: {X_test.shape}. Etiquetas alineadas.')

Train: (5634, 21); test: (1409, 21). Etiquetas alineadas.


## 2. Decisiones de codificación

Usamos **One-Hot completo para `PaymentMethod`**: cada método conserva su identidad sin imponer distancias ni un orden artificial. Agrupar métodos por su tasa de churn requeriría aprender del objetivo y podría introducir fuga de información. También aplicamos One-Hot completo a las demás nominales, incluidas las binarias. No eliminamos una categoría de referencia; la interpretación de coeficientes deberá considerar la redundancia entre indicadores.

Para `Contract` fijamos `Month-to-month < One year < Two year`, representados por 0, 1 y 2. El orden expresa duración y compromiso contractual; no implica que el efecto sobre churn sea lineal. El modelo posterior deberá reconocer que esta codificación introduce pasos numéricos iguales.

Detectamos las columnas a partir de entrenamiento. Las columnas numéricas, incluidas las calculadas, se conservan sin escalar para la tarea 23.

In [2]:
orden_contrato = ['Month-to-month', 'One year', 'Two year']
columnas_nominales = X_train.select_dtypes(include=['object', 'string', 'category']).columns.drop('Contract').tolist()
columnas_numericas = X_train.select_dtypes(include='number').columns.tolist()
assert set(columnas_nominales + ['Contract'] + columnas_numericas) == set(X_train.columns)
assert set(X_train['Contract']) == set(orden_contrato)

display(pd.DataFrame([
    {'tratamiento': 'One-Hot completo', 'columnas': ', '.join(columnas_nominales)},
    {'tratamiento': 'Ordinal 0, 1, 2', 'columnas': 'Contract'},
    {'tratamiento': 'Sin escalar', 'columnas': ', '.join(columnas_numericas)},
]))

,tratamiento,columnas
0,One-Hot completo,"gender, SeniorCitizen, Partner, Dependents, Ph..."
1,"Ordinal 0, 1, 2",Contract
2,Sin escalar,"tenure, MonthlyCharges, TotalCharges, ingreso_..."


La tabla documenta qué tratamiento recibe cada predictor. Los nombres y categorías de las nominales se aprenden exclusivamente en train. Un valor nominal desconocido en test produce ceros en el bloque de esa variable (`handle_unknown='ignore'`); un contrato desconocido provoca un error para revisar el dato, porque no tiene una posición definida en el orden.

## 3. Ajuste en train y transformación de test

Esta celda define el bloque de codificación que se puede integrar en el `ColumnTransformer` de la tarea 23. Aquí dejamos las numéricas como `passthrough`; en esa tarea se reemplazará ese bloque por el escalador. El único `fit_transform` recibe entrenamiento, mientras que test pasa solo por `transform`.

In [3]:
codificador = ColumnTransformer(
    transformers=[
        ('nominal', OneHotEncoder(handle_unknown='ignore', drop=None, sparse_output=False), columnas_nominales),
        ('contrato', OrdinalEncoder(categories=[orden_contrato], handle_unknown='error'), ['Contract']),
        ('numericas', 'passthrough', columnas_numericas),
    ],
    remainder='drop',
    verbose_feature_names_out=True,
)

train_codificado = codificador.fit_transform(X_train)
test_codificado = codificador.transform(X_test)
nombres = codificador.get_feature_names_out()
X_train_codificado = pd.DataFrame(train_codificado, columns=nombres, index=X_train.index)
X_test_codificado = pd.DataFrame(test_codificado, columns=nombres, index=X_test.index)
print(f'Train codificado: {X_train_codificado.shape}; test codificado: {X_test_codificado.shape}')
display(X_train_codificado.head())

Train codificado: (5634, 46); test codificado: (1409, 46)


,nominal__gender_Female,nominal__gender_Male,nominal__SeniorCitizen_No,nominal__SeniorCitizen_Yes,nominal__Partner_No,nominal__Partner_Yes,nominal__Dependents_No,nominal__Dependents_Yes,nominal__PhoneService_No,nominal__PhoneService_Yes,...,nominal__PaymentMethod_Bank transfer (automatic),nominal__PaymentMethod_Credit card (automatic),nominal__PaymentMethod_Electronic check,nominal__PaymentMethod_Mailed check,contrato__Contract,numericas__tenure,numericas__MonthlyCharges,numericas__TotalCharges,numericas__ingreso_estimado,numericas__diferencia_facturacion
fila_origen,,,,,,,,,,,,,,,,,,,,,
3738,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,35.0,49.20,1701.65,1722.00,-20.35
3151,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,15.0,75.10,1151.55,1126.50,25.05
4860,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,1.0,2.0,13.0,40.55,590.35,527.15,63.20
3867,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,2.0,26.0,73.50,1905.70,1911.00,-5.30
3810,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,1.0,44.55,44.55,44.55,0.00


Cada indicador nominal vale 0 o 1; `contrato__Contract` conserva el orden definido y las cinco columnas numéricas retienen sus valores. La expansión crea más columnas, pero conserva los mismos clientes y su correspondencia con `y_train` e `y_test`. Los datos todavía requieren escalado antes de cerrar el preprocesamiento.

## 4. Verificación de categorías, orden y ausencia de texto

Contrastamos las categorías aprendidas con train, comprobamos los indicadores y verificamos el contrato contra una equivalencia explícita. Controlamos que todas las columnas sean numéricas y finitas, que las numéricas originales no hayan cambiado y que las etiquetas sigan alineadas. La auditoría también informa si test contiene categorías nominales desconocidas, sin volver a ajustar el codificador.

In [4]:
onehot = codificador.named_transformers_['nominal']
for columna, categorias in zip(columnas_nominales, onehot.categories_):
    assert set(categorias) == set(X_train[columna].unique())
assert codificador.named_transformers_['contrato'].categories_[0].tolist() == orden_contrato

for original, convertido, objetivo in [
    (X_train, X_train_codificado, y_train),
    (X_test, X_test_codificado, y_test),
]:
    assert convertido.index.equals(objetivo.index)
    assert convertido.columns.is_unique
    assert all(pd.api.types.is_numeric_dtype(tipo) for tipo in convertido.dtypes)
    assert np.isfinite(convertido.to_numpy()).all()
    nominales = convertido.filter(like='nominal__')
    assert nominales.isin([0, 1]).all().all()
    esperado = original['Contract'].map(dict(zip(orden_contrato, [0, 1, 2])))
    np.testing.assert_array_equal(convertido['contrato__Contract'], esperado)
    for columna in columnas_numericas:
        np.testing.assert_array_equal(convertido[f'numericas__{columna}'], original[columna])

desconocidas = pd.Series({
    columna: int((~X_test[columna].isin(categorias)).sum())
    for columna, categorias in zip(columnas_nominales, onehot.categories_)
}, name='filas_test_con_categoria_desconocida')
display(desconocidas.to_frame())
print('Verificaciones correctas: orden contractual, categorías de train y matrices numéricas sin nulos.')

,filas_test_con_categoria_desconocida
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0
OnlineBackup,0
DeviceProtection,0


Verificaciones correctas: orden contractual, categorías de train y matrices numéricas sin nulos.


La auditoría permite detectar casos que no estaban representados en entrenamiento. No se agregan categorías aprendidas de test: se respeta el vocabulario de train. La comprobación del orden contractual verifica explícitamente que 0 significa mensual, 1 un año y 2 dos años.

## 5. Exportación y continuidad

Guardamos únicamente los predictores codificados en una subcarpeta intermedia para no ocupar las rutas finales de la tarea 23. Leemos de nuevo los archivos y verificamos su contenido. Las etiquetas siguen disponibles en `data/processed/particion/`.

Para la tarea 23, integrar este esquema con el escalador en un único `preprocessor`, ajustado sobre `X_train` original. Evitar aplicar de nuevo One-Hot a los CSV ya codificados. Las rutas finales `data/processed/X_train.csv`, `X_test.csv`, `y_train.csv` y `y_test.csv` quedan reservadas para esa tarea.

In [5]:
salida = RAIZ / 'data' / 'processed' / 'codificacion'
salida.mkdir(parents=True, exist_ok=True)
for nombre, datos in [('X_train', X_train_codificado), ('X_test', X_test_codificado)]:
    ruta = salida / f'{nombre}.csv'
    datos.to_csv(ruta, index=True, index_label='fila_origen')
    recuperado = pd.read_csv(ruta, index_col='fila_origen')
    pd.testing.assert_frame_equal(recuperado, datos, check_dtype=False)
print('Matrices codificadas exportadas y verificadas en:', salida)

Matrices codificadas exportadas y verificadas en: D:\src\tpi-analisis-de-datos\data\processed\codificacion


**Resultado:** predictores sin texto y contrato ordenado. La decisión para `PaymentMethod` queda documentada como One-Hot completo. No se ajustaron transformaciones sobre test ni se usó `Churn` para determinar categorías. El escalado y la exportación final corresponden a la tarea 23.